In [1]:
import pandas as pd
import requests
import os

In [2]:
def show_error(code, url):
    print("Error", code, "for",url)

In [3]:
def download_gkg(url):
    pieces = url.split("/")
    path = "data/" + pieces[-1]
    r = requests.get(url)
    if r.status_code == 200:
        with open(path, "wb") as f:
            f.write(r.content)
        if os.path.getsize(path) != len(r.content):
            print("Size mismatch for", path, "Expected", len(r.content), "got", os.path.getsize(path))
    else:
        show_error(r.status_code, url)

In [4]:
slots = pd.date_range("2024-10-16 08:00", "2024-10-16 10:00", freq="15min")

In [5]:
def gkg_url(date):
    return "http://data.gdeltproject.org/gdeltv2/" + date.strftime("%Y%m%d%H%M%S") +  ".gkg.csv.zip"

In [6]:
for slot in slots:
    download_gkg(gkg_url(slot))

In [7]:
gkg = pd.read_csv("data/20241016083000.gkg.csv.zip", sep="\t", header=None, dtype=str, quoting=3)
print(gkg.shape)
print(gkg[1].nunique(), gkg[1].iloc[0])

(1662, 27)
1 20241016083000


In [8]:
row = gkg.iloc[0]
for i, val in enumerate(row):
    print(i, len(str(val)), str(val)[:80])

0 16 20241016083000-0
1 14 20241016083000
2 1 1
3 18 lasvegasweekly.com
4 52 https://lasvegasweekly.com/events/2024/oct/16/57381/
5 3 nan
6 3 nan
7 3 nan
8 3 nan
9 3 nan
10 3 nan
11 9 las vegas
12 13 Las Vegas,429
13 3 nan
14 3 nan
15 88 6.93069306930693,9.9009900990099,2.97029702970297,12.8712871287129,21.7821782178
16 3 nan
17 5451 wc:80,c1.2:2,c12.1:6,c12.10:11,c12.12:6,c12.13:3,c12.14:4,c12.5:6,c12.7:6,c12.8:
18 79 https://media.lasvegasweekly.com/img/events/2024/10/Need-for-Speed-1200x628.jpg
19 3 nan
20 3 nan
21 3 nan
22 3 nan
23 66 Every Monday,13;South Point,354;South Point Yeti,415;Las Vegas,469
24 73 500,points on slots,55;100,dollars free play,204;400,Unisex Pullover,275;
25 3 nan
26 258 <PAGE_LINKS>https://lasvegasweekly.com/events/2024/oct/16/57381/%20https://south


In [9]:
print(gkg[13].head(10))

0                                                  NaN
1       nottingham university;university of nottingham
2                                                  NaN
3                                                  NaN
4                                  campaign collection
5          insurance technology market;qurate research
6    co louth;commission for the location of victim...
7    kerala high court on wednesday;high court;oppo...
8    parliament labour mp kim leadbeater;royal comm...
9                                          white house
Name: 13, dtype: str


In [10]:
print(gkg[13].notna().sum())

1202


In [11]:
print(gkg[26].iloc[0])

<PAGE_LINKS>https://lasvegasweekly.com/events/2024/oct/16/57381/%20https://southpointcasino.com/casino/slots/the-need-for-speed-south-point-400-giveaway/</PAGE_LINKS><PAGE_TITLE>Events Calendar - The Need For Speed: South Point 400 Gift Giveaway</PAGE_TITLE>


In [12]:
mask = gkg[13].str.contains("lvmh",na=False)
print(mask.sum())

0


In [13]:
for file in os.listdir("data"):
    df = pd.read_csv("data/" + file, sep="\t", header=None, dtype=str, quoting=3)
    mask_org = df[13].str.contains("lvmh", na=False)
    mask_title = df[26].str.contains("lvmh", na=False, case=False)
    print(file, mask_org.sum(), mask_title.sum())

20241016080000.gkg.csv.zip 0 1
20241016081500.gkg.csv.zip 0 3
20241016083000.gkg.csv.zip 0 1
20241016084500.gkg.csv.zip 0 0
20241016090000.gkg.csv.zip 0 0
20241016091500.gkg.csv.zip 0 1
20241016093000.gkg.csv.zip 0 1
20241016094500.gkg.csv.zip 0 0
20241016100000.gkg.csv.zip 0 1


In [14]:
df = pd.read_csv("data/20241016081500.gkg.csv.zip", sep="\t", header=None, dtype=str, quoting=3)
mask_title = df[26].str.contains("lvmh", na=False, case=False)

for text in df[26][mask_title]:
    print(text)
    print()

for orgs in df[13][mask_title]:
    print(orgs)

<PAGE_LINKS>https://www.thisismoney.co.uk/money/markets/article-13963701/Luxury-giant-LVMH-suffers-quarterly-sales-drop-pandemic.html</PAGE_LINKS><PAGE_AUTHORS>Live Commentary</PAGE_AUTHORS><PAGE_PRECISEPUBTIMESTAMP>20241016064700</PAGE_PRECISEPUBTIMESTAMP><PAGE_TITLE>BUSINESS LIVE: Inflation slows to 1.7%; Whitbread eyes &#xA3;300m profit boost; Just Eat hit by US slump</PAGE_TITLE>

<PAGE_LINKS>https://www.thisismoney.co.uk/money/markets/article-13963701/Luxury-giant-LVMH-suffers-quarterly-sales-drop-pandemic.html</PAGE_LINKS><PAGE_AUTHORS>Live Commentary</PAGE_AUTHORS><PAGE_PRECISEPUBTIMESTAMP>20241016064700</PAGE_PRECISEPUBTIMESTAMP><PAGE_ALTURL_AMP>https://www.thisismoney.co.uk/money/markets/article-13965119/amp/BUSINESS-LIVE-Inflation-slows-Whitbread-eyes-profit-boost-Just-Eat-hit-slump.html</PAGE_ALTURL_AMP><PAGE_TITLE>BUSINESS LIVE: Inflation slows to 1.7%; Whitbread eyes &#xA3;300m profit boost; Just Eat hit by US slump</PAGE_TITLE>

<PAGE_AUTHORS>MarketScreener</PAGE_AUTHORS>

## Findings: GDELT GKG exploration (16 Oct 2024, 08:00–10:00 UTC)

Test case: LVMH released its Q3 sales on 15 Oct 2024 after the Paris close, and the stock fell sharply on 16 Oct.

- **Organizations column (13) is unreliable**: 0 matches for "lvmh" in 9 files, even for an article titled "LVMH Results Cast Cloud on Luxury Stocks" (GDELT only extracted Burberry and Swatch).
- **Column 26 mixes titles and links**: 8 matches in total, but in the 08:15 file only 1 out of 3 was a real LVMH headline. The other 2 matched a URL.
- **Duplicates**: the same page can appear twice.
- **Exact timestamp**: `<PAGE_PRECISEPUBTIMESTAMP>` gives the publication time in UTC, needed to avoid look-ahead bias.
- **Sparse coverage**: very few articles per company, even on a major news day.

### Decisions
1. Match companies on `<PAGE_TITLE>` only, with a list of name variants per company.
2. Remove duplicate articles.
3. Measure coverage on ~10 random days before choosing the modelling approach (stock level on news days, CAC 40 index level, or French-language sources).